In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score,
                             roc_curve, classification_report)

df = pd.read_csv('titanic_cleaned.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,HasCabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,0


In [2]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')
print(df['Title'].value_counts())

def map_title(t):
    if t in ['Mr', 'Miss', 'Mrs', 'Master']:
        return t
    elif t in ['Mlle', 'Ms']:
        return 'Miss'
    elif t in ['Mme']:
        return 'Mrs'
    else:
        return 'Rare'

df['Title'] = df['Title'].apply(map_title)
print(df['Title'].value_counts())

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64
Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64


In [3]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [4]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked', 'Title'], drop_first=True)

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'HasCabin',
            'FamilySize', 'IsAlone'] + \
           [c for c in df.columns if c.startswith('Embarked_') or c.startswith('Title_')]
print(features)

['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'HasCabin', 'FamilySize', 'IsAlone', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare']


In [5]:
X = df[features]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

Train shape: (712, 15) Test shape: (179, 15)


In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)
y_pred_lr = logreg.predict(X_test_scaled)
y_proba_lr = logreg.predict_proba(X_test_scaled)[:, 1]

In [7]:
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

In [8]:
cv_scores_lr = cross_val_score(logreg, X_train_scaled, y_train, cv=5, scoring='accuracy')
print('Logistic Regression CV scores:', cv_scores_lr)
print('Mean CV accuracy:', cv_scores_lr.mean())

cv_scores_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')
print('Random Forest CV scores:', cv_scores_rf)
print('Mean CV accuracy:', cv_scores_rf.mean())

Logistic Regression CV scores: [0.8041958  0.7972028  0.84507042 0.81690141 0.82394366]
Mean CV accuracy: 0.8174628188712696
Random Forest CV scores: [0.7972028  0.78321678 0.82394366 0.83802817 0.81690141]
Mean CV accuracy: 0.81185856397124


In [9]:
print('=== Logistic Regression ===')
print('Accuracy:', accuracy_score(y_test, y_pred_lr))
print('Precision:', precision_score(y_test, y_pred_lr))
print('Recall:', recall_score(y_test, y_pred_lr))
print('F1:', f1_score(y_test, y_pred_lr))
print('ROC-AUC:', roc_auc_score(y_test, y_proba_lr))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_lr))

print('\n=== Random Forest ===')
print('Accuracy:', accuracy_score(y_test, y_pred_rf))
print('Precision:', precision_score(y_test, y_pred_rf))
print('Recall:', recall_score(y_test, y_pred_rf))
print('F1:', f1_score(y_test, y_pred_rf))
print('ROC-AUC:', roc_auc_score(y_test, y_proba_rf))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_rf))

=== Logistic Regression ===
Accuracy: 0.8379888268156425
Precision: 0.803030303030303
Recall: 0.7681159420289855
F1: 0.7851851851851852
ROC-AUC: 0.8715415019762845
Confusion Matrix:
 [[97 13]
 [16 53]]

=== Random Forest ===
Accuracy: 0.8268156424581006
Precision: 0.8166666666666667
Recall: 0.7101449275362319
F1: 0.7596899224806202
ROC-AUC: 0.8451910408432147
Confusion Matrix:
 [[99 11]
 [20 49]]


In [10]:
coefs = pd.Series(logreg.coef_[0], index=features).sort_values(ascending=False)
print(coefs)

importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

Sex           0.522254
HasCabin      0.383517
Fare          0.103666
Embarked_Q    0.075290
Title_Mrs    -0.069732
IsAlone      -0.125832
Embarked_S   -0.133807
Parch        -0.145863
FamilySize   -0.295499
Title_Rare   -0.295625
SibSp        -0.327427
Title_Miss   -0.407439
Age          -0.465806
Pclass       -0.678180
Title_Mr     -1.301782
dtype: float64
Title_Mr      0.185714
Sex           0.184411
Fare          0.128875
Age           0.095015
Pclass        0.082656
HasCabin      0.063425
Title_Miss    0.058782
Title_Mrs     0.054352
FamilySize    0.051428
SibSp         0.026823
Parch         0.020320
Embarked_S    0.017902
IsAlone       0.013603
Title_Rare    0.009177
Embarked_Q    0.007515
dtype: float64
